In [ ]:
import numpy as np
import ssqpy
from time import time, sleep

np.set_printoptions(suppress=True)

ssqpy.setSilentMode()

In [ ]:
from pycloudpendulumbenchmark import ControllerBase, MotorState

class SSQPYController(ControllerBase):
    def __init__(self):
        self.dt = 0.01
        MPC_H = 20
        
        V_WGT = 3e-2
        U_WGT = 2e-3
        TIP_WGT = 4.2
        J1_WGT = 0.0
        J2_WGT = 2.0
        
        self.torque_clip = 0.1
        
        self.model = ssqpy.model.Model(
            MPC_H,
            self.dt,
            urdf_path="pendubot.urdf",
            actuated_joints=[0],
            solver_mode=ssqpy.model.SolverMode.InverseDynamics
        )
        
        self.nq = self.model.getnq()
        self.nv = self.model.getnv()
        self.nu = self.model.getnu()
        
        vel_cost = ssqpy.model.costs.SquaredJointVelocityCost(self.model, V_WGT)
        u_cost = ssqpy.model.costs.SquaredControlCost(self.model, U_WGT)
        tip_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
            self.model, "tip", np.array((0.0, 0.0, 0.2)), TIP_WGT
        )
        config_cost = ssqpy.model.costs.SquaredConfigurationErrorCost(
            self.model, np.array((np.pi, 0.0)), np.array((J1_WGT, J2_WGT))
        )
        
        for k in range(MPC_H):
            self.model.addCost(k, vel_cost)
            self.model.addCost(k, u_cost)
            self.model.addCost(k, tip_cost)
            self.model.addCost(k, config_cost)
        
        self.model.addCost(MPC_H, vel_cost)
        self.model.addCost(MPC_H, tip_cost)
        self.model.addCost(MPC_H, config_cost)
        
        ssqp_params = ssqpy.solvers.ssqpParams()
        ssqp_params.tolerance = 1e-1
        
        admm_params = ssqpy.solvers.admmParams()
        admm_params.abs_tolerance = 1e-2
        admm_params.rel_tolerance = 1e-2
        admm_params.warm_start = True
        ssqp_params.admmParams = admm_params
        
        
        self.mpc = ssqpy.solvers.MPC(
            self.model, ssqp_params, sqp_iters=4, qp_iters=100
        )

    def get_control_output(self, state):
        mq = np.array([state[i].position for i in range(len(state))])
        mv = np.array([state[i].velocity for i in range(len(state))])

        try:
            u = self.mpc.step(np.hstack((mq, mv)))[1].stage(0)
        except RuntimeError:
            u = np.zeros(self.nv)

        tau = self.model.inverseDynamics(mq, mv, u)
        tau = np.clip(tau, -self.torque_clip, self.torque_clip)

        return [tau[0]]

In [ ]:
from pycloudpendulumbenchmark import ControlLoop
from pycloudpendulumbenchmark.evaluators import GoalHeightEvaluator

if __name__ == "__main__":
    controller = SSQPYController()
    evaluator = GoalHeightEvaluator()

    with open("../token.txt", "r") as f:
        token = f.readlines()[0].strip()
    
    control_loop = ControlLoop(
        cell_id=203,
        initial_position=np.array((0.0, 0.0)),
        disturbances=[],
        dt=0.001,
        user_token = token,
        experiment_time = 20.0,
        experiment_type = "Pendubot"
    )
    control_loop.start()

    while not control_loop.finished():
        state = control_loop.get_state()
        control_loop_time = control_loop.time()

        controller_output = controller.get_control_output(state)
        evaluator.evaluate(state, controller_output, control_loop_time)

        control_loop.step(controller_output)
    
    video_url= control_loop.stop()

    score = evaluator.get_score()
    print("Score:", score)